# Clean training features (silence + exact CQT duplicates)

Slice `training/features/training.npz` (do not overwrite it). Drop:

1. 16 digital-silence `C_diminished_4-{48..63}` clips (all-zero CQT)
2. The extra member of each exact CQT-MD5 pair (keep the lexicographically first `class/file`)

`dataset-info.ipynb` already found 13 hash groups / 40 members: the silence cluster plus 12 `A#_major_4` pairs. Residual near-dups and the 391 waveform `flag_mostly_silent` rows are **not** dropped.

Writes `training/features/training-clean.npz` (`features`, `labels`, `files`, `orig_index`) and `results/excluded.csv`.

## Config

In [1]:
from pathlib import Path
import hashlib
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display, Markdown


def find_training_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "features").is_dir() and (cand / "datasets").is_dir():
            return cand
        nested = cand / "training"
        if (nested / "features").is_dir() and (nested / "datasets").is_dir():
            return nested
    raise FileNotFoundError(f"training root not found from {start}")


TRAINING_ROOT = find_training_root(Path.cwd())
FEATURES_IN = TRAINING_ROOT / "features" / "training.npz"
FEATURES_OUT = TRAINING_ROOT / "features" / "training-clean.npz"
AUDIO_DIR = TRAINING_ROOT / "datasets" / "training"
RESULTS_DIR = TRAINING_ROOT / "notebooks" / "cnn-latest" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXCLUDED_CSV = RESULTS_DIR / "excluded.csv"
EXPECTED_N = 7200
EXPECTED_SILENT = 16
EXPECTED_DUP_GROUPS = 13
EXPECTED_DUP_MEMBERS = 40
EXPECTED_KEEP = 7172

cfg = pd.DataFrame([
    {"key": "cwd", "value": str(Path.cwd())},
    {"key": "training_root", "value": str(TRAINING_ROOT)},
    {"key": "features_in", "value": str(FEATURES_IN)},
    {"key": "features_out", "value": str(FEATURES_OUT)},
    {"key": "excluded_csv", "value": str(EXCLUDED_CSV)},
    {"key": "expected_keep", "value": EXPECTED_KEEP},
])
display(cfg)
assert FEATURES_IN.is_file(), FEATURES_IN
assert AUDIO_DIR.is_dir(), AUDIO_DIR

,key,value
0,cwd,/home/seya/code/chord-detection/training/noteb...
1,training_root,/home/seya/code/chord-detection/training
2,features_in,/home/seya/code/chord-detection/training/featu...
3,features_out,/home/seya/code/chord-detection/training/featu...
4,excluded_csv,/home/seya/code/chord-detection/training/noteb...
5,expected_keep,7172


## Load features and reconstruct file keys

In [2]:
data = np.load(FEATURES_IN, allow_pickle=True)
features = data["features"]
labels = data["labels"].astype(str)
assert features.shape == (EXPECTED_N, 216, 188), features.shape
assert len(labels) == EXPECTED_N

files, recon = [], []
for cls in sorted(p.name for p in AUDIO_DIR.iterdir() if p.is_dir()):
    for name in sorted(p.name for p in (AUDIO_DIR / cls).iterdir() if p.is_file()):
        files.append(f"{cls}/{name}")
        recon.append(cls)
files = np.asarray(files)
recon = np.asarray(recon)
assert len(files) == EXPECTED_N
assert np.array_equal(recon, labels), "dataset walk does not match npz labels"

inv = pd.DataFrame({
    "orig_index": np.arange(EXPECTED_N),
    "file": files,
    "label": labels,
})
display(Markdown(f"Loaded **{len(inv)}** rows from `{FEATURES_IN.name}`"))
display(inv.groupby("label").size().rename("n").reset_index().head())

Loaded **7200** rows from `training.npz`

,label,n
0,A#_diminished_4,200
1,A#_major_4,200
2,A#_minor_4,200
3,A_diminished_4,200
4,A_major_4,200


## Digital silence and exact CQT MD5 groups

In [3]:
flat = features.reshape(len(features), -1)
silent_mask = np.abs(flat).max(axis=1) == 0
silent_idx = np.where(silent_mask)[0]
assert len(silent_idx) == EXPECTED_SILENT, len(silent_idx)
assert set(labels[silent_idx]) == {"C_diminished_4"}

silent_df = inv.loc[silent_idx, ["orig_index", "file", "label"]].reset_index(drop=True)
display(Markdown(f"### Digital silence (**{len(silent_df)}**)"))
display(silent_df)

def row_hash(row: np.ndarray) -> str:
    return hashlib.md5(np.ascontiguousarray(row).tobytes()).hexdigest()

hashes = np.array([row_hash(flat[i]) for i in range(len(flat))])
inv["cqt_hash"] = hashes
hash_counts = inv["cqt_hash"].value_counts()
dup_hashes = hash_counts[hash_counts > 1]
assert len(dup_hashes) == EXPECTED_DUP_GROUPS, len(dup_hashes)
assert int(dup_hashes.sum()) == EXPECTED_DUP_MEMBERS, int(dup_hashes.sum())

groups = (
    inv[inv.cqt_hash.isin(dup_hashes.index)]
    .groupby("cqt_hash")
    .agg(n=("file", "size"), classes=("label", lambda s: ", ".join(sorted(set(s)))), files=("file", lambda s: ", ".join(sorted(s))))
    .sort_values("n", ascending=False)
    .reset_index()
)
display(Markdown("### Exact CQT duplicate groups"))
display(groups[["n", "classes", "files"]])

### Digital silence (**16**)

,orig_index,file,label
0,2543,C_diminished_4/C_diminished_4-48.ogg,C_diminished_4
1,2544,C_diminished_4/C_diminished_4-49.ogg,C_diminished_4
2,2546,C_diminished_4/C_diminished_4-50.ogg,C_diminished_4
3,2547,C_diminished_4/C_diminished_4-51.ogg,C_diminished_4
4,2548,C_diminished_4/C_diminished_4-52.ogg,C_diminished_4
5,2549,C_diminished_4/C_diminished_4-53.ogg,C_diminished_4
6,2550,C_diminished_4/C_diminished_4-54.ogg,C_diminished_4
7,2551,C_diminished_4/C_diminished_4-55.ogg,C_diminished_4
8,2552,C_diminished_4/C_diminished_4-56.ogg,C_diminished_4
9,2553,C_diminished_4/C_diminished_4-57.ogg,C_diminished_4


### Exact CQT duplicate groups

,n,classes,files
0,16,C_diminished_4,"C_diminished_4/C_diminished_4-48.ogg, C_dimini..."
1,2,A#_major_4,"A#_major_4/A#_major_4-169.ogg, A#_major_4/A#_m..."
2,2,A#_major_4,"A#_major_4/A#_major_4-121.ogg, A#_major_4/A#_m..."
3,2,A#_major_4,"A#_major_4/A#_major_4-3.ogg, A#_major_4/A#_maj..."
4,2,A#_major_4,"A#_major_4/A#_major_4-125.ogg, A#_major_4/A#_m..."
5,2,A#_major_4,"A#_major_4/A#_major_4-12.ogg, A#_major_4/A#_ma..."
6,2,A#_major_4,"A#_major_4/A#_major_4-77.ogg, A#_major_4/A#_ma..."
7,2,A#_major_4,"A#_major_4/A#_major_4-7.ogg, A#_major_4/A#_maj..."
8,2,A#_major_4,"A#_major_4/A#_major_4-159.ogg, A#_major_4/A#_m..."
9,2,A#_major_4,"A#_major_4/A#_major_4-10.ogg, A#_major_4/A#_ma..."


## File-byte MD5 (report only)

In [4]:
file_hashes = []
for key in files:
    raw = (AUDIO_DIR / key).read_bytes()
    file_hashes.append(hashlib.md5(raw).hexdigest())
inv["file_md5"] = file_hashes
file_counts = inv["file_md5"].value_counts()
file_dup = file_counts[file_counts > 1]
extra_file_dups = inv[inv.file_md5.isin(file_dup.index) & ~inv.cqt_hash.isin(dup_hashes.index)]

display(Markdown(
    f"File-byte MD5 groups with copies: **{len(file_dup)}** "
    f"({int(file_dup.sum())} files). Extra hits not already in the CQT-dup list: **{len(extra_file_dups)}**."
))
if len(file_dup):
    display(
        inv[inv.file_md5.isin(file_dup.index)]
        .groupby("file_md5")
        .agg(n=("file", "size"), files=("file", lambda s: ", ".join(sorted(s))))
        .sort_values("n", ascending=False)
        .reset_index()
    )
if len(extra_file_dups):
    display(Markdown("### Extra file-byte dups (not dropped)"))
    display(extra_file_dups[["orig_index", "file", "label", "file_md5"]].reset_index(drop=True))

File-byte MD5 groups with copies: **16** (40 files). Extra hits not already in the CQT-dup list: **0**.

,file_md5,n,files
0,9e3386e29c54cd3398ba53291b4bc61f,6,"C_diminished_4/C_diminished_4-51.ogg, C_dimini..."
1,1dcbed031f9626ec34612c33c7e7d61d,4,"C_diminished_4/C_diminished_4-49.ogg, C_dimini..."
2,23abe425af9a1c0bb52c42613204755f,4,"C_diminished_4/C_diminished_4-48.ogg, C_dimini..."
3,30438429283f2f781fab28545f9d750b,2,"C_diminished_4/C_diminished_4-52.ogg, C_dimini..."
4,4ae6c9af8f6cd899d7276c39dbe81f0c,2,"A#_major_4/A#_major_4-10.ogg, A#_major_4/A#_ma..."
5,4f907daac7b4b108e8e8996f83c175c5,2,"A#_major_4/A#_major_4-82.ogg, A#_major_4/A#_ma..."
6,50b531807e06908c74e451a3dfed3608,2,"A#_major_4/A#_major_4-159.ogg, A#_major_4/A#_m..."
7,5f99bef3144931d753b9d834ff9d55f3,2,"A#_major_4/A#_major_4-121.ogg, A#_major_4/A#_m..."
8,77e2c156077c858902ab3078b0dbf104,2,"A#_major_4/A#_major_4-12.ogg, A#_major_4/A#_ma..."
9,8cb1ed238cab2aca2e30fea260a138ea,2,"A#_major_4/A#_major_4-125.ogg, A#_major_4/A#_m..."


## Keep mask and write `training-clean.npz`

In [5]:
drop_rows = []
keep = np.ones(EXPECTED_N, dtype=bool)

for i in silent_idx:
    keep[i] = False
    drop_rows.append({
        "orig_index": int(i),
        "file": files[i],
        "label": labels[i],
        "cqt_hash": hashes[i],
        "reason": "silence",
    })

for h, g in inv.groupby("cqt_hash"):
    if len(g) < 2:
        continue
    if bool(g.index.isin(silent_idx).all()):
        continue
    ordered = g.sort_values("file")
    keeper = int(ordered.index[0])
    for i in ordered.index[1:]:
        if not keep[i]:
            continue
        keep[i] = False
        drop_rows.append({
            "orig_index": int(i),
            "file": files[i],
            "label": labels[i],
            "cqt_hash": hashes[i],
            "reason": "cqt_dup",
            "kept_file": files[keeper],
        })

excluded = pd.DataFrame(drop_rows).sort_values(["reason", "file"]).reset_index(drop=True)
keep_idx = np.where(keep)[0]
assert len(keep_idx) == EXPECTED_KEEP, (len(keep_idx), excluded.reason.value_counts().to_dict())
assert int((excluded.reason == "silence").sum()) == EXPECTED_SILENT
assert int((excluded.reason == "cqt_dup").sum()) == 12

excluded.to_csv(EXCLUDED_CSV, index=False)
np.savez_compressed(
    FEATURES_OUT,
    features=features[keep_idx],
    labels=labels[keep_idx],
    files=files[keep_idx],
    orig_index=keep_idx.astype(np.int32),
)

before = inv.groupby("label").size().rename("n_orig")
after = pd.Series(labels[keep_idx]).value_counts().rename("n_clean")
counts = pd.concat([before, after], axis=1).fillna(0).astype(int)
counts["dropped"] = counts["n_orig"] - counts["n_clean"]
changed = counts[counts.dropped > 0].sort_values("dropped", ascending=False)

display(Markdown(f"Wrote **{len(keep_idx)}** rows → `{FEATURES_OUT}`"))
display(Markdown(f"Excluded **{len(excluded)}** → `{EXCLUDED_CSV}`"))
display(excluded)
display(Markdown("### Class counts that changed"))
display(changed.reset_index().rename(columns={"index": "label"}))

Wrote **7172** rows → `/home/seya/code/chord-detection/training/features/training-clean.npz`

Excluded **28** → `/home/seya/code/chord-detection/training/notebooks/cnn-latest/results/excluded.csv`

,orig_index,file,label,cqt_hash,reason,kept_file
0,299,A#_major_4/A#_major_4-189.ogg,A#_major_4,5ddcdb9e6fc8d68c0c6d9315da87b195,cqt_dup,A#_major_4/A#_major_4-12.ogg
1,306,A#_major_4/A#_major_4-195.ogg,A#_major_4,adc5ebdc90a770dc558afa1475ac8773,cqt_dup,A#_major_4/A#_major_4-10.ogg
2,314,A#_major_4/A#_major_4-21.ogg,A#_major_4,cf9cd4892ef2f6d17ef17e67b40d4e40,cqt_dup,A#_major_4/A#_major_4-176.ogg
3,328,A#_major_4/A#_major_4-34.ogg,A#_major_4,079244d6e5e301564f280e594179445e,cqt_dup,A#_major_4/A#_major_4-169.ogg
4,348,A#_major_4/A#_major_4-52.ogg,A#_major_4,9d7a0bc4ca8e0a96c797df7abcfcb160,cqt_dup,A#_major_4/A#_major_4-159.ogg
5,360,A#_major_4/A#_major_4-63.ogg,A#_major_4,21752772f658ae20bbb13a04ec8ec645,cqt_dup,A#_major_4/A#_major_4-125.ogg
6,362,A#_major_4/A#_major_4-65.ogg,A#_major_4,1306673e59540a33c33768ed97da3494,cqt_dup,A#_major_4/A#_major_4-121.ogg
7,371,A#_major_4/A#_major_4-73.ogg,A#_major_4,dac432dd4b10eeb27a43e3a4fe8aeacd,cqt_dup,A#_major_4/A#_major_4-103.ogg
8,384,A#_major_4/A#_major_4-85.ogg,A#_major_4,dc396b7d82fbde326d3fd95e5fea69be,cqt_dup,A#_major_4/A#_major_4-82.ogg
9,388,A#_major_4/A#_major_4-89.ogg,A#_major_4,1ba0c1aad74bfa320641685710a16d91,cqt_dup,A#_major_4/A#_major_4-3.ogg


### Class counts that changed

,label,n_orig,n_clean,dropped
0,C_diminished_4,200,184,16
1,A#_major_4,200,188,12
